In [2]:
import pandas as pd
import os
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import img_to_array
import numpy as np

# CSV 파일 로드
csv_path = r'C:\2024신한해커톤\운동 인식 프로젝트\CSV_Files/trainData_CSV.csv'
df = pd.read_csv(csv_path)

# 이미지 경로 설정
image_dir = r'C:\2024신한해커톤\운동 인식 프로젝트\Data\013.피트니스자세_sample\원천데이터/'

# 이미지와 레이블을 리스트에 저장
images = []
labels = []

for index, row in df.iterrows():
    img_path = os.path.join(image_dir, row['img_key'])
    img = Image.open(img_path)
    img = img.resize((224, 224))  # 이미지 크기 조정 (모델 입력 크기에 맞게)
    img = img_to_array(img)
    images.append(img)
    labels.append(row['active'])

# NumPy 배열로 변환
images = np.array(images, dtype='float32') / 255.0
labels = np.array(labels)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\2024신한해커톤\\운동 인식 프로젝트\\Data\\013.피트니스자세_sample\\원천데이터\\033-1-1-21-Z17_A-0000001.jpg'

In [4]:
import os
import pandas as pd
from PIL import Image
from tensorflow.keras.preprocessing.image import img_to_array
import numpy as np
from tqdm import tqdm  # tqdm 임포트

# CSV 파일 로드
csv_path = r'C:\2024신한해커톤\운동 인식 프로젝트\CSV_Files/trainData_CSV.csv'
df = pd.read_csv(csv_path)

# 상위 디렉토리 설정
base_dir = r'C:\2024신한해커톤\운동 인식 프로젝트\Data\013.피트니스자세_sample\원천데이터/'

# 이미지 파일 경로를 찾는 함수
def find_image_file(base_dir, img_key):
    for root, dirs, files in os.walk(base_dir):
        if img_key in files:
            return os.path.join(root, img_key)
    return None

# 이미지와 레이블을 리스트에 저장
images = []
labels = []

# tqdm을 사용하여 진행률 표시
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing images"):
    img_key = row['img_key']
    img_path = find_image_file(base_dir, img_key)
    
    if img_path:
        img = Image.open(img_path)
        img = img.resize((224, 224))  # 이미지 크기 조정 (모델 입력 크기에 맞게)
        img = img_to_array(img)
        images.append(img)
        labels.append(row['active'])
    else:
        print(f'이미지를 찾을 수 없습니다: {img_key}')

# NumPy 배열로 변환
images = np.array(images, dtype='float32') / 255.0
labels = np.array(labels)

# 데이터셋 분할 및 모델 학습 단계는 동일합니다.


Processing images:   8%|▊         | 737/9600 [08:25<1:41:24,  1.46it/s]


KeyboardInterrupt: 

In [12]:
import pandas as pd

# Load the CSV file


csv_path = r'C:\2024신한해커톤\운동 인식 프로젝트\CSV_Files/trainData_CSV.csv'


df = pd.read_csv(csv_path)

# Filter the rows where active is 1 and 0
active_1 = df[df['active'] == 1]
active_0 = df[df['active'] == 0]

# Randomly sample 50 rows from each
sample_active_1 = active_1.sample(n=500, random_state=1)
sample_active_0 = active_0.sample(n=500, random_state=1)

# Concatenate the samples into a new DataFrame
sampled_df = pd.concat([sample_active_1, sample_active_0])

# Save the new DataFrame to a CSV file
sampled_df.to_csv('sample.csv', index=False)


In [14]:
import pandas as pd
import shutil
import os
from tqdm import tqdm

# 추출된 CSV 파일 경로 (이미 추출된 샘플 데이터가 들어있는 CSV)
csv_file_path = r'C:\2024신한해커톤\운동 인식 프로젝트\CSV_Files/sample.csv'

# 이미지가 저장된 상위 디렉토리 경로
image_folder_path = r'C:\2024신한해커톤\운동 인식 프로젝트\Data\013.피트니스자세_sample\원천데이터'

# 결과를 저장할 폴더 경로
output_folder_path = r'C:\2024신한해커톤\운동 인식 프로젝트\Data\test_data'

# CSV 파일 로드
df = pd.read_csv(csv_file_path)

# 각 폴더 경로 생성
os.makedirs(os.path.join(output_folder_path, '1'), exist_ok=True)
os.makedirs(os.path.join(output_folder_path, '0'), exist_ok=True)

# 이미지 파일 경로를 찾기 위해 상위 디렉토리 내 모든 파일 탐색
all_files = []
for root, dirs, files in os.walk(image_folder_path):
    for file in files:
        all_files.append(os.path.join(root, file))

# 이미지 키에 해당하는 이미지를 active 값에 따라 해당 폴더에 복사
for index, row in tqdm(df.iterrows(), total=len(df), desc="Copying images"):
    img_key = row['img_key']
    active_value = row['active']
    matching_files = [file for file in all_files if img_key in file]
    if matching_files:
        destination_folder = os.path.join(output_folder_path, str(active_value))
        shutil.copy2(matching_files[0], os.path.join(destination_folder, os.path.basename(matching_files[0])))
    else:
        print(f"Image {img_key} not found.")

print("Images have been copied successfully.")



Copying images: 100%|██████████| 1000/1000 [00:08<00:00, 118.52it/s]

Images have been copied successfully.


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# 원본 데이터 폴더 경로
image_folder_path = r'C:\2024신한해커톤\운동 인식 프로젝트\Data\test_data'

# 1과 0에 해당하는 이미지 경로 설정
data_1_dir = os.path.join(image_folder_path, '1')
data_0_dir = os.path.join(image_folder_path, '0')

# 1인 이미지와 0인 이미지 파일 리스트 얻기
images_1 = [os.path.join(data_1_dir, img) for img in os.listdir(data_1_dir)]
images_0 = [os.path.join(data_0_dir, img) for img in os.listdir(data_0_dir)]

# 각각의 이미지에 레이블 부여
labels_1 = [1] * len(images_1)
labels_0 = [0] * len(images_0)

# 이미지 경로와 레이블을 데이터프레임으로 결합
df_1 = pd.DataFrame({'filename': images_1, 'label': labels_1})
df_0 = pd.DataFrame({'filename': images_0, 'label': labels_0})

# 데이터프레임 합치기
df = pd.concat([df_1, df_0], ignore_index=True)

# train/test split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=1)


In [18]:
# 'label' 열을 문자열로 변환
train_df['label'] = train_df['label'].astype(str)
val_df['label'] = val_df['label'].astype(str)

In [20]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 이미지 데이터 전처리
train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

# 학습 데이터 생성기
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col='filename',
    y_col='label',
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary'
)

# 검증 데이터 생성기
validation_generator = validation_datagen.flow_from_dataframe(
    val_df,
    x_col='filename',
    y_col='label',
    target_size=(150, 150),
    batch_size=20,
    class_mode='binary'
)

# 모델 구축
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# 모델 컴파일
model.compile(loss='binary_crossentropy',
              optimizer=tf.keras.optimizers.Adam(),
              metrics=['accuracy'])

# 모델 학습
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=50,
    validation_data=validation_generator,
    validation_steps=len(validation_generator)
)

Found 800 validated image filenames belonging to 2 classes.
Found 200 validated image filenames belonging to 2 classes.


c:\Users\NHS\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50


c:\Users\NHS\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


40/40 ━━━━━━━━━━━━━━━━━━━━ 62s 1s/step - accuracy: 0.4898 - loss: 0.7436 - val_accuracy: 0.5000 - val_loss: 0.6934
Epoch 2/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 974us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 3/50


c:\Users\NHS\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)


40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.4693 - loss: 0.6943 - val_accuracy: 0.5200 - val_loss: 0.6931
Epoch 4/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 375us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 5/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.5063 - loss: 0.6932 - val_accuracy: 0.4800 - val_loss: 0.6930
Epoch 6/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 475us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 7/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.5486 - loss: 0.6924 - val_accuracy: 0.4950 - val_loss: 0.6932
Epoch 8/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 375us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 9/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 60s 1s/step - accuracy: 0.5228 - loss: 0.6935 - val_accuracy: 0.5000 - val_loss: 0.6957
Epoch 10/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 425us/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 11/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.5121 - loss: 0.6954 - val_accuracy: 0.4850 - val_loss: 0.6931
Epo

In [21]:
# 모델 학습 후 최고의 모델을 저장
model.save('best_model.h5')  # .h5 형식으로 저장
model.save('best_model.keras')  # .keras 형식으로 저장

In [30]:
from tensorflow.keras.preprocessing import image
import numpy as np

# 예측할 이미지 경로
img_path = r'C:\2024신한해커톤\운동 인식 프로젝트\Data\testImage\rightImage.jpg'

# 이미지를 모델에 넣기 위한 전처리
img = image.load_img(img_path, target_size=(150, 150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)  # 배치 크기 추가
img_array /= 255.0  # 모델이 학습된 대로 이미지 스케일 조정

# 이미지 예측
prediction = model.predict(img_array)

print(prediction[0])
print(prediction[0][0])

# 예측 결과 출력
if prediction[0] > 0.5:
    print(f'자세가 올바릅니다 , 예측값은 {prediction[0][0]:.2f} 입니다.')
else:
    print(f'자세가 올바르지않습니다 , 예측값은 {prediction[0][0]:.2f} 입니다.')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
[0.00326469]
0.0032646905
자세가 올바르지않습니다 , 예측값은 0.00 입니다.
